# torch.compile() for LLM Inference Optimization

**Stage 1: Basic Optimization**

This notebook demonstrates `torch.compile()`, PyTorch 2.0's graph compilation feature that provides:
- **1.2-1.5x speedup** for inference (varies by model and hardware)
- **Automatic kernel fusion** - combines operations for efficiency
- **Zero code changes** - just wrap your model

## What is torch.compile()?

`torch.compile()` is PyTorch 2.0's JIT compiler that:
1. **Captures computation graph** during first execution
2. **Optimizes graph** - fuses operations, removes redundancy
3. **Generates efficient kernels** - tailored to your hardware
4. **Caches compiled code** for subsequent runs

### Compilation Modes

| Mode | Speed | Compilation Time | Use Case |
|------|-------|------------------|----------|
| **default** | Good | Fast | General purpose |
| **reduce-overhead** | Better | Medium | Inference with small models |
| **max-autotune** | Best | Slow | Production (compile once) |

### When torch.compile() Helps Most
- Small to medium models (< 7B parameters)
- Models with many small operations
- GPU inference (CPU benefits are smaller)
- Production deployments (amortize compilation cost)

### When It May Not Help
- Very large models (compilation overhead)
- Dynamic shapes or control flow
- First request (compilation adds latency)

**Reference**: See `LLM_INFERENCE_ROADMAP.md` - Stage 1, Basic Optimizations

In [ ]:
# Install required packages
!pip install torch>=2.0 transformers accelerate matplotlib pandas numpy -q

In [ ]:
import torch
import torch._dynamo as dynamo
from transformers import AutoModelForCausalLM, AutoTokenizer
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from typing import List, Dict
import gc
import warnings

warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Check if torch.compile is available
if torch.__version__ < '2.0.0':
    print("\n⚠ WARNING: torch.compile() requires PyTorch 2.0+")
    print("  Please upgrade: pip install torch>=2.0")
else:
    print("✓ torch.compile() is available")

if device == "cuda":
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")

## 1. Basic torch.compile() Usage

In [ ]:
# Load model
model_name = "gpt2"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
model.eval()

if device == "cpu":
    model = model.to(device)

print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

# Compile model (default mode)
print("\nCompiling model with torch.compile()...")
if torch.__version__ >= '2.0.0':
    model_compiled = torch.compile(model, mode="default")
    print("✓ Model compiled (default mode)")
else:
    model_compiled = model
    print("✗ Using eager mode (PyTorch < 2.0)")

In [ ]:
# Simple comparison: eager vs compiled
def benchmark_model(model, tokenizer, prompt: str, max_new_tokens: int = 50, num_runs: int = 5):
    """Benchmark a model."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    
    # Warmup (especially important for compiled model)
    print("  Warming up...", end=" ")
    for _ in range(3):
        with torch.no_grad():
            _ = model.generate(input_ids, max_new_tokens=10, use_cache=True)
    print("done")
    
    if device == "cuda":
        torch.cuda.synchronize()
    
    # Benchmark
    print(f"  Running {num_runs} iterations...", end=" ")
    times = []
    
    for _ in range(num_runs):
        start = time.time()
        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                use_cache=True,
                do_sample=False,
            )
        
        if device == "cuda":
            torch.cuda.synchronize()
        
        times.append(time.time() - start)
    
    print("done")
    
    avg_time = np.mean(times)
    std_time = np.std(times)
    tokens_per_sec = max_new_tokens / avg_time
    
    return {
        'avg_time': avg_time,
        'std_time': std_time,
        'tokens_per_sec': tokens_per_sec,
        'times': times,
        'output': tokenizer.decode(outputs[0], skip_special_tokens=True),
    }


prompt = "The future of artificial intelligence includes"
max_tokens = 100

print("\n" + "="*80)
print("Benchmarking: Eager vs Compiled")
print("="*80)

# Benchmark eager mode
print("\nEager mode (baseline):")
eager_result = benchmark_model(model, tokenizer, prompt, max_tokens, num_runs=5)
print(f"  Average time: {eager_result['avg_time']:.3f}s (±{eager_result['std_time']:.3f}s)")
print(f"  Throughput: {eager_result['tokens_per_sec']:.1f} tokens/sec")

# Benchmark compiled mode
if torch.__version__ >= '2.0.0':
    print("\nCompiled mode (torch.compile):")
    compiled_result = benchmark_model(model_compiled, tokenizer, prompt, max_tokens, num_runs=5)
    print(f"  Average time: {compiled_result['avg_time']:.3f}s (±{compiled_result['std_time']:.3f}s)")
    print(f"  Throughput: {compiled_result['tokens_per_sec']:.1f} tokens/sec")
    
    speedup = eager_result['avg_time'] / compiled_result['avg_time']
    print(f"\n" + "="*80)
    print(f"Speedup: {speedup:.2f}x")
    print(f"Throughput gain: {(speedup - 1) * 100:.1f}%")
else:
    print("\n✗ Skipping compiled benchmark (PyTorch < 2.0)")

## 2. Comparing Compilation Modes

In [ ]:
if torch.__version__ >= '2.0.0':
    # Load fresh models for each mode
    compilation_modes = ['default', 'reduce-overhead', 'max-autotune']
    
    mode_results = []
    
    print("\n" + "="*80)
    print("Comparing Compilation Modes")
    print("="*80)
    
    # Benchmark eager first
    print("\n1. Eager (no compilation):")
    eager_bench = benchmark_model(model, tokenizer, prompt, max_tokens=50, num_runs=3)
    mode_results.append({
        'Mode': 'eager',
        'Time (s)': eager_bench['avg_time'],
        'Tokens/sec': eager_bench['tokens_per_sec'],
        'Speedup': 1.0,
    })
    print(f"  Time: {eager_bench['avg_time']:.3f}s")
    print(f"  Throughput: {eager_bench['tokens_per_sec']:.1f} tok/s")
    
    # Benchmark each compilation mode
    for i, mode in enumerate(compilation_modes, start=2):
        print(f"\n{i}. Compiled mode: {mode}")
        print(f"  Compiling... (this may take a while for '{mode}')")
        
        # Clear previous compilation
        if device == "cuda":
            torch.cuda.empty_cache()
        
        # Compile with specific mode
        model_mode = torch.compile(model, mode=mode)
        
        # Benchmark
        result = benchmark_model(model_mode, tokenizer, prompt, max_tokens=50, num_runs=3)
        
        speedup = eager_bench['avg_time'] / result['avg_time']
        
        mode_results.append({
            'Mode': mode,
            'Time (s)': result['avg_time'],
            'Tokens/sec': result['tokens_per_sec'],
            'Speedup': speedup,
        })
        
        print(f"  Time: {result['avg_time']:.3f}s")
        print(f"  Throughput: {result['tokens_per_sec']:.1f} tok/s")
        print(f"  Speedup vs eager: {speedup:.2f}x")
        
        # Clean up
        del model_mode
        gc.collect()
    
    # Display results
    df_modes = pd.DataFrame(mode_results)
    
    print("\n" + "="*80)
    print("Summary: Compilation Modes")
    print("="*80)
    print(df_modes.to_string(index=False))
    
    # Find best mode
    best_mode = df_modes.loc[df_modes['Speedup'].idxmax()]
    print(f"\nBest mode: {best_mode['Mode']} ({best_mode['Speedup']:.2f}x speedup)")
else:
    print("Skipping compilation mode comparison (PyTorch < 2.0)")
    df_modes = None

In [ ]:
# Visualize compilation modes
if torch.__version__ >= '2.0.0' and df_modes is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Throughput comparison
    colors = ['coral', 'lightblue', 'lightgreen', 'plum']
    bars = axes[0].bar(df_modes['Mode'], df_modes['Tokens/sec'], color=colors[:len(df_modes)])
    
    axes[0].set_ylabel('Throughput (tokens/sec)', fontsize=12)
    axes[0].set_title('Throughput by Compilation Mode', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Annotate bars
    for bar, val in zip(bars, df_modes['Tokens/sec']):
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.1f}',
                    ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Speedup
    bars2 = axes[1].bar(df_modes['Mode'], df_modes['Speedup'], color=colors[:len(df_modes)])
    axes[1].axhline(y=1, color='red', linestyle='--', alpha=0.5, label='Baseline (eager)')
    
    axes[1].set_ylabel('Speedup vs Eager', fontsize=12)
    axes[1].set_title('Speedup by Compilation Mode', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Annotate speedup
    for bar, val in zip(bars2, df_modes['Speedup']):
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.2f}x',
                    ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('compilation_modes.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nKey Insights:")
    print("1. max-autotune provides best performance (but slowest compilation)")
    print("2. reduce-overhead is good for small models / low latency")
    print("3. default is a balanced choice for most cases")
    print("4. Speedup varies by model, hardware, and workload")

## 3. Compilation Time vs Runtime Speedup

In [ ]:
if torch.__version__ >= '2.0.0':
    def measure_compilation_time(model, tokenizer, mode: str, prompt: str):
        """Measure time to compile model."""
        
        # Clear cache
        dynamo.reset()
        if device == "cuda":
            torch.cuda.empty_cache()
        
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        
        # Compile
        start = time.time()
        model_compiled = torch.compile(model, mode=mode)
        
        # First run triggers compilation
        with torch.no_grad():
            _ = model_compiled.generate(input_ids, max_new_tokens=20, use_cache=True)
        
        if device == "cuda":
            torch.cuda.synchronize()
        
        compilation_time = time.time() - start
        
        return compilation_time, model_compiled
    
    
    print("\n" + "="*80)
    print("Measuring Compilation Time")
    print("="*80)
    
    compilation_results = []
    
    for mode in ['default', 'reduce-overhead', 'max-autotune']:
        print(f"\nCompiling with mode='{mode}'...")
        comp_time, model_comp = measure_compilation_time(model, tokenizer, mode, prompt)
        
        print(f"  Compilation time: {comp_time:.2f}s")
        
        # Measure runtime speedup
        result = benchmark_model(model_comp, tokenizer, prompt, max_tokens=50, num_runs=3)
        speedup = eager_bench['avg_time'] / result['avg_time']
        
        # Calculate break-even point
        # How many inferences needed to amortize compilation cost?
        time_saved_per_inference = eager_bench['avg_time'] - result['avg_time']
        break_even_requests = comp_time / time_saved_per_inference if time_saved_per_inference > 0 else float('inf')
        
        compilation_results.append({
            'Mode': mode,
            'Compilation Time (s)': comp_time,
            'Runtime Speedup': speedup,
            'Break-even (requests)': break_even_requests,
        })
        
        print(f"  Runtime speedup: {speedup:.2f}x")
        print(f"  Break-even after: {break_even_requests:.0f} requests")
        
        del model_comp
        gc.collect()
    
    df_compilation = pd.DataFrame(compilation_results)
    
    print("\n" + "="*80)
    print("Compilation Time Analysis")
    print("="*80)
    print(df_compilation.to_string(index=False))
else:
    print("Skipping compilation time analysis (PyTorch < 2.0)")
    df_compilation = None

In [ ]:
# Visualize break-even analysis
if torch.__version__ >= '2.0.0' and df_compilation is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Compilation time
    bars = axes[0].bar(df_compilation['Mode'], df_compilation['Compilation Time (s)'], 
                      color=['lightblue', 'lightgreen', 'coral'])
    
    axes[0].set_ylabel('Compilation Time (seconds)', fontsize=12)
    axes[0].set_title('First-Run Compilation Time', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, df_compilation['Compilation Time (s)']):
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.1f}s',
                    ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Break-even requests
    bars2 = axes[1].bar(df_compilation['Mode'], df_compilation['Break-even (requests)'], 
                       color=['lightblue', 'lightgreen', 'coral'])
    
    axes[1].set_ylabel('Number of Requests', fontsize=12)
    axes[1].set_title('Break-even Point (Requests to Amortize Compilation)', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars2, df_compilation['Break-even (requests)']):
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.0f}',
                    ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('compilation_breakeven.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nKey Insights:")
    print("1. Compilation time increases with more aggressive optimization")
    print("2. max-autotune takes longest but provides best runtime speedup")
    print("3. Break-even point is typically < 100 requests for production")
    print("4. Always compile for production deployments (one-time cost)")

## 4. Graph Breaks and How to Avoid Them

Graph breaks occur when torch.compile() can't trace through certain operations, forcing it to fall back to eager mode.

In [ ]:
if torch.__version__ >= '2.0.0':
    # Enable graph break debugging
    import torch._dynamo.config
    
    print("Common Causes of Graph Breaks:")
    print("="*80)
    print("\n1. Data-dependent control flow")
    print("   Example: if tensor.item() > 0: ...")
    print("   Fix: Avoid .item() calls, use torch operations")
    
    print("\n2. Dynamic shapes")
    print("   Example: Variable sequence lengths without padding")
    print("   Fix: Use padding or dynamic=True in torch.compile()")
    
    print("\n3. Python print/debug statements")
    print("   Example: print(f'Value: {tensor}')")
    print("   Fix: Remove or guard with torch.compiler.is_compiling()")
    
    print("\n4. Unsupported operations")
    print("   Example: Some custom CUDA kernels")
    print("   Fix: Check PyTorch docs for supported ops")
    
    print("\n" + "="*80)
    print("\nHow to Debug Graph Breaks:")
    print("="*80)
    
    debug_code = '''
import torch._dynamo as dynamo

# Enable verbose logging
dynamo.config.verbose = True
dynamo.config.log_level = "INFO"

# Compile model
model_compiled = torch.compile(model, mode="default")

# Run inference - watch for "Graph break" messages
output = model_compiled(input_ids)

# Explain graph breaks
dynamo.explain(model)(input_ids)
'''
    print(debug_code)
    
    print("\n" + "="*80)
    print("\nBest Practices to Minimize Graph Breaks:")
    print("="*80)
    
    practices = [
        "Use standard PyTorch operations (avoid custom Python logic)",
        "Pad sequences to fixed lengths for batching",
        "Avoid .item(), .tolist(), or other tensor-to-Python conversions",
        "Remove debug print statements from hot paths",
        "Use torch.compile() at the highest level possible",
        "Test with dynamo.explain() to identify issues",
        "Consider fullgraph=True to error on breaks",
    ]
    
    for i, practice in enumerate(practices, 1):
        print(f"  {i}. {practice}")
else:
    print("Skipping graph break analysis (PyTorch < 2.0)")

## 5. Production Deployment Considerations

In [ ]:
# Production implementation
production_code = '''
# Production inference with torch.compile()

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

class CompiledInferenceEngine:
    """Production inference engine with torch.compile()."""
    
    def __init__(
        self,
        model_name: str,
        compile_mode: str = "max-autotune",  # Best for production
        device: str = "cuda",
    ):
        self.device = device
        
        # Load model with optimal precision
        dtype = torch.float16 if device == "cuda" else torch.float32
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        print(f"Loading model: {model_name}")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            device_map="auto" if device == "cuda" else None,
        )
        self.model.eval()
        
        if device == "cpu":
            self.model = self.model.to(device)
        
        # Compile model
        if torch.__version__ >= '2.0.0':
            print(f"Compiling model with mode={compile_mode}...")
            self.model = torch.compile(self.model, mode=compile_mode)
            print("Compilation complete")
        else:
            print("torch.compile() not available (PyTorch < 2.0)")
        
        # Warmup: trigger compilation
        print("Warming up (triggering compilation)...")
        dummy_input = self.tokenizer("warmup", return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            _ = self.model.generate(dummy_input, max_new_tokens=10)
        print("Ready for inference")
    
    def generate(self, prompt: str, max_new_tokens: int = 50, **kwargs):
        """Generate text for a single prompt."""
        input_ids = self.tokenizer(prompt, return_tensors="pt").input_ids.to(self.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                use_cache=True,  # Always use KV cache
                pad_token_id=self.tokenizer.eos_token_id,
                **kwargs
            )
        
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    def generate_batch(self, prompts: list, max_new_tokens: int = 50, **kwargs):
        """Generate text for multiple prompts (batched)."""
        inputs = self.tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                use_cache=True,
                pad_token_id=self.tokenizer.eos_token_id,
                **kwargs
            )
        
        return self.tokenizer.batch_decode(outputs, skip_special_tokens=True)


# Usage
if __name__ == "__main__":
    # Initialize engine (compilation happens here)
    engine = CompiledInferenceEngine(
        model_name="gpt2",
        compile_mode="max-autotune",  # Best performance for production
        device="cuda",
    )
    
    # Generate (fast after warmup)
    prompt = "The future of AI is"
    output = engine.generate(prompt, max_new_tokens=50)
    print(f"Prompt: {prompt}")
    print(f"Output: {output}")
    
    # Batch inference
    prompts = [
        "Artificial intelligence will",
        "Machine learning enables",
        "Deep neural networks can",
    ]
    outputs = engine.generate_batch(prompts, max_new_tokens=30)
    
    for p, o in zip(prompts, outputs):
        print(f"\nPrompt: {p}")
        print(f"Output: {o}")
'''

print("Production Inference with torch.compile()")
print("="*100)
print(production_code)
print("="*100)

In [ ]:
print("\n\nProduction Deployment Checklist")
print("="*80)

checklist = [
    ("✓ Use PyTorch 2.0+ for torch.compile() support", "Upgrade if needed: pip install torch>=2.0"),
    ("✓ Choose compilation mode based on use case", "max-autotune for production, default for dev"),
    ("✓ Compile model during initialization", "One-time cost, amortized over many requests"),
    ("✓ Always include warmup before serving", "Triggers compilation, ensures first request is fast"),
    ("✓ Combine with KV caching", "torch.compile() doesn't replace KV cache"),
    ("✓ Combine with FP16/BF16", "Mixed precision + compilation = best performance"),
    ("✓ Use static batching when possible", "Fixed shapes compile better"),
    ("✓ Monitor for graph breaks", "Use dynamo.explain() during development"),
    ("✓ Test thoroughly before production", "Ensure outputs match eager mode"),
    ("✓ Consider torch.jit.save() for model artifacts", "Save compiled model for faster startup"),
]

for item, detail in checklist:
    print(f"\n{item}")
    print(f"  → {detail}")

print("\n" + "="*80)
print("\nWhen to Use torch.compile():")
print("  ✓ Production deployments (amortize compilation cost)")
print("  ✓ Small to medium models (< 7B parameters)")
print("  ✓ GPU inference (benefits are larger)")
print("  ✓ Fixed/static input shapes")
print("  ✓ After other optimizations (KV cache, FP16, batching)")

print("\nWhen NOT to Use torch.compile():")
print("  ✗ Development/debugging (graph breaks hide errors)")
print("  ✗ Single-request scenarios (compilation overhead not amortized)")
print("  ✗ Highly dynamic models (frequent graph breaks)")
print("  ✗ Very large models (compilation time may be prohibitive)")

## 6. Complete Benchmark: Combining All Optimizations

In [ ]:
if torch.__version__ >= '2.0.0':
    print("\n" + "="*80)
    print("Complete Optimization Stack Benchmark")
    print("="*80)
    print("\nComparing:")
    print("  1. Baseline (FP32, no cache, no compile)")
    print("  2. + KV Cache")
    print("  3. + FP16")
    print("  4. + torch.compile()")
    print("  5. + Static Batching (batch_size=4)")
    
    optimization_results = []
    
    # Test prompts
    test_prompt = "The future of artificial intelligence"
    batch_prompts = [test_prompt] * 4
    
    # 1. Baseline (would need FP32 model without cache)
    # For demonstration, we'll use relative improvements
    
    # Load FP16 model
    model_fp16 = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    model_fp16.eval()
    
    # Benchmark configurations
    print("\nRunning benchmarks...")
    
    # Config 1: FP16 + KV Cache (our baseline for this demo)
    print("\n1. FP16 + KV Cache (baseline)")
    result1 = benchmark_model(model_fp16, tokenizer, test_prompt, max_tokens=50, num_runs=3)
    optimization_results.append({
        'Configuration': 'FP16 + KV Cache',
        'Time (s)': result1['avg_time'],
        'Tokens/sec': result1['tokens_per_sec'],
        'Speedup': 1.0,
    })
    baseline_time = result1['avg_time']
    print(f"  Time: {result1['avg_time']:.3f}s, Throughput: {result1['tokens_per_sec']:.1f} tok/s")
    
    # Config 2: + torch.compile()
    print("\n2. FP16 + KV Cache + torch.compile()")
    model_compiled_fp16 = torch.compile(model_fp16, mode="default")
    result2 = benchmark_model(model_compiled_fp16, tokenizer, test_prompt, max_tokens=50, num_runs=3)
    optimization_results.append({
        'Configuration': '+ torch.compile()',
        'Time (s)': result2['avg_time'],
        'Tokens/sec': result2['tokens_per_sec'],
        'Speedup': baseline_time / result2['avg_time'],
    })
    print(f"  Time: {result2['avg_time']:.3f}s, Throughput: {result2['tokens_per_sec']:.1f} tok/s")
    print(f"  Speedup: {baseline_time / result2['avg_time']:.2f}x")
    
    # Config 3: + Static Batching
    print("\n3. FP16 + KV Cache + torch.compile() + Batching (4x)")
    start = time.time()
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        _ = model_compiled_fp16.generate(**inputs, max_new_tokens=50, use_cache=True)
    if device == "cuda":
        torch.cuda.synchronize()
    batch_time = time.time() - start
    
    per_request_time = batch_time / 4
    batch_throughput = (4 * 50) / batch_time
    
    optimization_results.append({
        'Configuration': '+ Batching (4x)',
        'Time (s)': per_request_time,
        'Tokens/sec': batch_throughput,
        'Speedup': baseline_time / per_request_time,
    })
    print(f"  Time per request: {per_request_time:.3f}s, Throughput: {batch_throughput:.1f} tok/s")
    print(f"  Speedup: {baseline_time / per_request_time:.2f}x")
    
    # Display results
    df_optimizations = pd.DataFrame(optimization_results)
    
    print("\n" + "="*80)
    print("Optimization Stack Results")
    print("="*80)
    print(df_optimizations.to_string(index=False))
    
    # Cumulative speedup
    total_speedup = optimization_results[-1]['Speedup']
    print(f"\n→ Total speedup from all optimizations: {total_speedup:.2f}x")
    print(f"  (Note: This is relative to FP16+KV baseline, not true FP32 baseline)")
    
    # Clean up
    del model_fp16, model_compiled_fp16
    gc.collect()
else:
    print("Skipping optimization stack benchmark (PyTorch < 2.0)")

## Summary and Key Takeaways

### Performance Results
- **Speedup**: 1.2-1.5x for most models (varies by hardware and model)
- **Compilation time**: 10-60 seconds (one-time cost)
- **Break-even**: < 100 requests for production workloads

### Compilation Modes
1. **default**: Balanced performance and compilation time
2. **reduce-overhead**: Lower overhead, good for small models
3. **max-autotune**: Best performance, longest compilation (production choice)

### When torch.compile() Works Best
- Production deployments (amortize compilation cost)
- Small to medium models (< 7B parameters)
- GPU inference (larger benefits than CPU)
- Static input shapes (avoid graph breaks)
- After other optimizations (KV cache, FP16, batching)

### Common Pitfalls
1. **Graph breaks**: Data-dependent control flow, dynamic shapes
2. **First request latency**: Compilation happens on first run
3. **Memory overhead**: Compiled code uses more memory
4. **Debugging**: Harder to debug than eager mode

### Production Best Practices
1. Use `max-autotune` mode for production
2. Always include warmup to trigger compilation
3. Combine with KV cache, FP16, and batching
4. Monitor for graph breaks during development
5. Test outputs match eager mode before deployment
6. Document compilation settings in deployment configs

### Complete Optimization Stack
```
FP32 baseline
  + KV Cache         → 5-10x speedup
  + FP16/BF16        → 1.5-2x speedup
  + torch.compile()  → 1.2-1.5x speedup
  + Static Batching  → 2-4x throughput
  ─────────────────────────────────────
  Total:             → 15-100x improvement!
```

### Limitations
- Not a silver bullet (1.2-1.5x vs 5-10x for KV cache)
- Requires PyTorch 2.0+
- First request has compilation overhead
- May not work well with dynamic models

### Next Steps
- **Stage 1**: INT8 Quantization (Notebook 14) - Memory reduction
- **Stage 2**: Advanced optimizations (Flash Attention, PagedAttention)
- **Stage 3**: Serving frameworks (vLLM, TGI, TensorRT-LLM)

### References
- LLM_INFERENCE_ROADMAP.md - Stage 1: Basic Optimizations
- PyTorch 2.0 Docs: https://pytorch.org/get-started/pytorch-2.0/
- torch.compile() Tutorial: https://pytorch.org/tutorials/intermediate/torch_compile_tutorial.html
- Dynamo Deep Dive: https://pytorch.org/docs/stable/dynamo/index.html